# LST → SUHI (Surface Urban Heat Island) 계산

## 목적
- 연도별 여름철 LST 데이터를 이용해 셀 단위 SUHI 산출
- SUHI = LST − 참조(배경) LST (참조: 전역 하위 5% percentile)

## 입력 데이터
- `data/processed/lst/lst_{YEAR}_summer_100m_{CLOUD_THRESHOLD}.csv` (cell_id, LST_C) — **05_lst_outlier_check에서 이상치 제거한 데이터**

## 출력 데이터
- `data/processed/suhi/suhi_{YEAR}_summer_100m.csv` (cell_id, suhi)

In [1]:
import pandas as pd
import os

In [3]:
# =========================
# Global Configuration
# =========================

# LST 데이터 디렉터리 (05_lst_outlier_check.ipynb 산출물, 이상치 제거 후)
LST_DIR = "../data_processed/processed/lst"

# SUHII 출력 디렉터리
OUTPUT_DIR = "../data_processed/processed/suhii"

# 참조 LST percentile (하위 20% = 배경/식생·수계 등 상대적으로 낮은 온도)
REFERENCE_PERCENTILE = 20

# 기본 연도, 구름덮임 기준 (LST 파일명: lst_{YEAR}_summer_100m.csv)
DEFAULT_YEAR = 2025

In [7]:
# =========================
# SUHII 계산 함수
# =========================

def compute_suhii_and_save(
    year: int = DEFAULT_YEAR,
    lst_dir: str = LST_DIR,
    output_dir: str = OUTPUT_DIR,
    reference_percentile: float = REFERENCE_PERCENTILE,
) -> str:
    """
    지정 연도의 LST CSV를 읽어 SUHI를 계산하고, cell_id·suhi CSV를 저장합니다.

    SUHI = LST_C - reference_LST (reference_LST = 전역 LST의 reference_percentile 퍼센타일)

    Parameters
    ----------
    year : int
        분석 연도
    lst_dir : str
        LST CSV가 있는 디렉터리
    output_dir : str
        SUHI CSV 저장 디렉터리
    reference_percentile : float
        참조 LST로 쓸 percentile (기본 20 = 하위 20%)

    Returns
    -------
    str
        저장된 CSV 파일 경로
    """
    os.makedirs(output_dir, exist_ok=True)

    lst_path = os.path.join(
        lst_dir, f"lst_{year}_summer_100m_clean.csv"
    )
    if not os.path.isfile(lst_path):
        raise FileNotFoundError(
            f"LST 파일이 없습니다: {lst_path}\n"
            f"05_lst_outlier_check.ipynb에서 해당 연도·구름덮임으로 이상치 제거한 LST를 먼저 산출하세요."
        )

    df = pd.read_csv(lst_path)
    if "LST_C" not in df.columns or "cell_id" not in df.columns:
        raise ValueError(
            f"LST CSV에 cell_id, LST_C 컬럼이 필요합니다: {lst_path}"
        )

    ref_lst = df["LST_C"].quantile(reference_percentile / 100.0)
    df["suhii"] = df["LST_C"] - ref_lst

    out_name = f"suhii_{year}_summer_100m.csv"
    out_path = os.path.join(output_dir, out_name)
    df[["cell_id", "suhii"]].to_csv(
        out_path, index=False,
        encoding="utf-8-sig"
    )

    return out_path

In [8]:
# =========================
# 원하는 연도로 SUHII 계산 및 CSV 저장
# =========================

YEAR = 2025

out_path = compute_suhii_and_save(
    year=YEAR,
)